# 02 — Rules and Derivations

This notebook covers the reasoning layer:
- Rule DSL: `vars`, `Pred`, `Not`, `RuleRef`
- `sdk.run(rule)` execution
- Query DSL: `dict` and `instance` row formats
- Derivation: `evaluate` / `accept` / `accept_many`
- `Body` with confidence, views, optional ProbLog
- Provenance validation
- Audit APIs: `explain_fact`, `conflicts`
- SDKRegistry for rule/derivation versioning

**Prerequisites:** Run [01_sdk_basics.ipynb](01_sdk_basics.ipynb) to understand Entity/Store.  
**Next:** [03_certainty_and_evidence_tree.ipynb](03_certainty_and_evidence_tree.ipynb)

## 0. Setup

Re-create the schema and seed data from notebook 01.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent / "src"))

In [2]:
from __future__ import annotations
import shutil, tempfile, warnings
from pathlib import Path
from pprint import pprint, pp

from factpy_kernel.sdk import (
    SDKStore, SDKRegistry, Entity, Identity, Field,
    Rule, RuleRef, Query, Derivation, Pred, Not, Body, vars,
    SDKStoreError, SDKSchemaError,
)


class Country(Entity):
    iso_code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")

class Language(Entity):
    code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")

class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    aliases: str = Field(cardinality="multi")
    age: int = Field(cardinality="single")
    country: Country = Field(cardinality="single")
    tag: str = Field(cardinality="multi")

class LivesIn(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    user: User = Field(cardinality="single")
    country: Country = Field(cardinality="single")
    since: int = Field(cardinality="single")

class HasLanguage(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    country: Country = Field(cardinality="single")
    language: Language = Field(cardinality="single")

class Speaks(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    user: User = Field(cardinality="single")
    language: Language = Field(cardinality="single")

classes = [Country, Language, User, LivesIn, HasLanguage, Speaks]
sdk = SDKStore.from_schema_classes(classes, default_row_format="dict")

# Seed data
tx = sdk.batch(meta={"source": "seed"})
de = tx.entity(Country, iso_code="DE"); de.name.set("Germany")
cn = tx.entity(Country, iso_code="CN"); cn.name.set("China")
u1 = tx.entity(User, user_id="u-001", locale="zh")
u1.name.set("Alice"); u1.tag.add("vip"); u1.age.set(30); u1.country.set(de)
u2 = tx.entity(User, user_id="u-002").bind(locale="en")
u2.name.set("Bob"); u2.tag.add("staff"); u2.age.set(28); u2.country.set(cn)
tx.commit()
u1_ref = sdk.ref(User, user_id="u-001", locale="zh")
print("Seed complete")

Seed complete


## 1. Rule DSL

Rules use `vars`, `Pred`, `Not`, and `RuleRef` to express logic.

In [3]:
with vars("u", "vip", "n") as (u, vip, n):
    vip_rule = Rule(
        id="q.vip_not_blocked", version="1.0.0",
        select=[vip],
        where=[Pred("user:tag", u, vip), Not([Pred("user:tag", u, "blocked")])],
        expose=True,
    )

    name_rule = Rule(
        id="q.name", version="1.0.0",
        select=[n],
        where=[User(u), u.name == n],
    )

    vip_ref_rule = Rule(
        id="q.vip_ref", version="1.0.0",
        select=[u],
        where=[RuleRef(vip_rule)(u)],
    )

print("vip rows:", sdk.run(vip_rule, row_format="dict"))
print("name rows:", sdk.run(name_rule, row_format="dict"))
print("RuleRef rows:", sdk.run(vip_ref_rule, row_format="dict"))

vip rows: [{'vip': 'staff'}, {'vip': 'vip'}]
name rows: [{'n': 'Alice'}, {'n': 'Bob'}]
RuleRef rows: [{'u': 'staff'}, {'u': 'vip'}]


## 2. Query DSL

`dict` (default) and `instance` row formats.

In [4]:
with vars("u", "loc", "nm") as (u, loc, nm):
    q_dict = Query(
        head=[User(u), User.name(locale=loc, name=nm)],
        where=[User(u), u.locale == loc, u.name == nm],
        on_missing="error", on_type_mismatch="error",
    )

print("dict rows:")
pp(sdk.run(q_dict))

with vars("u", "loc") as (u, loc):
    q_instance = Query(
        head=User(u),
        where=[User(u), u.locale == loc, loc == "zh"],
    )

print("instance refs:", [r.ref for r in sdk.run(q_instance, row_format="instance")])

dict rows:
[{'u': EntitySnapshot(entity_type='User', ref='idref_v1:User:42uubjiy5t3miixsx5pm5dmpc5ix37lluxopopkiyv4e37i3rk7q', age=28, aliases=(), country='idref_v1:Cou...vwlzkfquddcfa', locale='en', name='Bob', tag=('staff',), user_id='u-002'),
  'nm': 'Bob'},
 {'u': EntitySnapshot(entity_type='User', ref='idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba', age=30, aliases=(), country='idref_v1:Cou...sx3rx53fk6qtq', locale='zh', name='Alice', tag=('vip',), user_id='u-001'),
  'nm': 'Alice'}]
instance refs: ['idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba']


## 3. Derivation: evaluate / accept

The full pipeline: define → evaluate → accept.

In [5]:
with vars("u", "loc", "nm") as (u, loc, nm):
    alias_drv = Derivation(
        id="drv.derived_tag", version="1.0.0",
        where=[User(u), u.locale == loc, u.name == nm],
        head=User.tag(locale=loc, tag=nm),
    )

cands = sdk.evaluate(alias_drv, mode="native")
print("candidates:", len(cands))

if cands:
    dry = sdk.accept(cands[0], approved_by="demo", note="apply", dry_run=True)
    print("dry run:", dry)

    applied = sdk.accept(cands[0], approved_by="demo", note="apply")
    print("applied:", applied)

    dup = sdk.accept(cands[0], approved_by="demo", note="apply")
    print("duplicate (idempotent):", dup)

candidates: 2
dry run: AcceptResult(run_id='21a5efa8331049539e9f559d41f27c6e', accepted_count=1, skipped_count=0, written_assertions=[{'asrt_id': '<dry_run>', 'pred_id': 'user:tag', 'key_tuple_digest': 'sha256:87075cae6df5cf476dead778fe00835bf0fa8e2723e226cca8821a85e5049633'}], skipped_reason_counts={}, diagnostics=[], diagnostics_contract_version=1, entity_ref=None, candidate_id='cand_v2:9b535514e66f9b8f9fbe6067c88e5eb6616307f797101467f5a138479e0c2e16', candidate_key='candk_v2:1357e45b5654a4a9bc1fdf057aebfdfb9e7544a655a58dd6d2b10e1fbb92f044')
applied: AcceptResult(run_id='21a5efa8331049539e9f559d41f27c6e', accepted_count=1, skipped_count=0, written_assertions=[{'asrt_id': '27db9390a7ae4fe6aac03ce3a6fbb467', 'pred_id': 'user:tag', 'key_tuple_digest': 'sha256:87075cae6df5cf476dead778fe00835bf0fa8e2723e226cca8821a85e5049633'}], skipped_reason_counts={}, diagnostics=[], diagnostics_contract_version=1, entity_ref=None, candidate_id='cand_v2:9b535514e66f9b8f9fbe6067c88e5eb6616307f797101467f

In [6]:
# Multi-head derivation + accept_many
with vars("u", "loc", "nm", "tg") as (u, loc, nm, tg):
    multi_drv = Derivation(
        id="drv.multi", version="1.0.0",
        where=[User(u), u.locale == loc, u.name == nm, u.tag == tg],
        head=[User.name(locale=loc, name=nm), User.tag(locale=loc, tag=tg)],
    )

multi_cands = sdk.evaluate(multi_drv, mode="native")
print("multi-head candidates:", len(multi_cands))

if multi_cands:
    many = sdk.accept_many(multi_cands[:2], mode="atomic")
    print("accept_many:", many)

multi-head candidates: 5
accept_many: [{'candidate_id': 'cand_v2:d1094f64e17d3f1ae90e0e1b5765227dafe730fcd6b5124245be578b81150d80', 'candidate_key': 'candk_v2:3a9383b22ec683e8ac84463089053d8ff9d922f404727d8f30526fd091acfcae', 'state': 'ACCEPTED', 'entity_ref': None, 'error': None}, {'candidate_id': 'cand_v2:0ff5efb13ae93c324094adefc76b45da8f1aeccf4cb0bc5a85110fa45afef56b', 'candidate_key': 'candk_v2:80edfe03e534824db025069cb4f288a320253856936f63c5aac2be4d2ee16bf2', 'state': 'ACCEPTED', 'entity_ref': None, 'error': None}]


### 3.1 Body, Views, and Optional ProbLog

In [7]:
from factpy_kernel.core.store.types import ViewSpec

# Write confidence values
sdk.add(User.tag, u1_ref, "vip", meta={"source": "profile", "confidence": 0.82})
sdk.add(User.tag, u1_ref, "vip", meta={"source": "model", "confidence": 0.64})

# Body with confidence
with vars("u", "loc", "tg") as (u, loc, tg):
    body_drv = Derivation(
        id="drv.body_demo", version="1.0.0",
        where=[
            Body([User(u), u.locale == loc, Pred("user:tag", u, tg)], confidence=0.9),
            Body([User(u), Pred("user:tag", u, tg)], confidence=0.6),
        ],
        head=User.tag(locale=loc, tag=tg),
    )

print("body candidates:", len(sdk.evaluate(body_drv, mode="native")))

# View with confidence strategy
sdk.views.create("mean_conf", ViewSpec(confidence_strategy="mean"))
rows, meta = sdk.run(vip_rule, view="mean_conf", return_display_meta=True, row_format="dict")
print("view rows:", len(rows), "display_meta:", meta[0] if meta else None)

# Optional ProbLog
try:
    import factpy_kernel.adapters.problog
    print("problog candidates:", len(sdk.evaluate(body_drv, mode="problog")))
except Exception as exc:
    print("problog skipped:", type(exc).__name__, exc)

body candidates: 3
view rows: 3 display_meta: {'confidence': None, 'confidence_strategy': 'mean', 'source_breakdown': []}
problog skipped: ProbLogEngineError ProbLog CLI is not available: problog


## 4. Provenance Validation

In [8]:
report_ok = sdk.validate_provenance(
    {"derived_rule_id": "drv.derived_tag", "derived_rule_version": "1.0.0",
     "run_id": "run-001", "support_kind": "tuple",
     "support_digest": "sha256:" + "a" * 64},
    standard="derivation_v1",
)
report_bad = sdk.validate_provenance(
    {"derived_rule_id": "", "derived_rule_version": "1.0.0",
     "run_id": "", "support_kind": "tuple",
     "support_digest": "not-a-sha256"},
    standard="derivation_v1",
)
print("good:", report_ok.ok)
print("bad:", report_bad.ok)
pp(report_bad.errors)

good: True
bad: False
[{'code': 'provenance_required_field_missing_or_invalid',
  'severity': 'error',
  'path': '$.provenance.derived_rule_id',
  'message': 'derived_rule_id must be non-empty string',
  'data': {'key': 'derived_rule_id'}},
 {'code': 'provenance_required_field_missing_or_invalid',
  'severity': 'error',
  'path': '$.provenance.run_id',
  'message': 'run_id must be non-empty string',
  'data': {'key': 'run_id'}},
 {'code': 'provenance_required_digest_missing_or_invalid',
  'severity': 'error',
  'path': '$.provenance.support_digest',
  'message': "support_digest must be 'sha256:<hex>'",
  'data': {'key': 'support_digest'}}]


## 5. Audit: explain_fact / conflicts

In [9]:
def pred_id_for(sdk_obj, owner_type, field_name):
    for pred in sdk_obj.schema_ir.get("predicates", []):
        if not isinstance(pred, dict): continue
        if pred.get("owner_type") == owner_type and pred.get("py_field_name") == field_name:
            return pred["pred_id"]
    raise RuntimeError(f"not found: {owner_type}.{field_name}")

user_tag_pred = pred_id_for(sdk, "User", "tag")
print("explain_fact:")
pprint(sdk.explain_fact(user_tag_pred, u1_ref, "vip"))
print("\nconflicts:")
pprint(sdk.conflicts(user_tag_pred, u1_ref))

explain_fact:
{'active_claims': [{'args': ('idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba',
                             'vip'),
                    'asrt_id': '0d6857b5e7ae45db9618c6b857d79cda',
                    'meta': {'ingested_at': 1774829991964428000,
                             'source': 'seed'}},
                   {'args': ('idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba',
                             'vip'),
                    'asrt_id': '84d658e9f4584f89a4474ade51e7d2eb',
                    'meta': {'ingested_at': 1774829992009545000,
                             'source': 'profile'}},
                   {'args': ('idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba',
                             'vip'),
                    'asrt_id': '231dbfe729224942975a52aeb4e54543',
                    'meta': {'ingested_at': 1774829992010360000,
                             'source': 'model'}}],
 'chosen_asrt_id': '0d6857b5e7

## 6. SDKRegistry

In [10]:
registry_root = Path(tempfile.gettempdir()) / "factpy_registry_demo"
if registry_root.exists():
    shutil.rmtree(registry_root)

registry = SDKRegistry(root_dir=registry_root)
registry.apply_schema_classes(classes)
registry.register_rule(vip_rule, schema_ir=sdk.schema_ir)
registry.register_derivation(alias_drv, schema_ir=sdk.schema_ir)

print("rule ids:", registry.list_rule_ids())
print("derivation ids:", registry.list_derivation_ids())

rule ids: ['q.vip_not_blocked']
derivation ids: ['drv.derived_tag']


---
**Next:** [03_certainty_and_evidence_tree.ipynb](03_certainty_and_evidence_tree.ipynb) — Certainty propagation and evidence tree deep dive